# Qwen single-agent TEC tool-calling batch experiment


## 1. Clean Colab Setup

This notebook is a small experimental stand for checking the LLM single-agent across five TEC task scenarios. It is not a production benchmark dataset.

- Run the clean setup cell first.
- After it finishes, use Runtime -> Restart runtime.
- Then continue from the import check cell.
- Change only the CONFIG cell for normal experiments.
- This notebook does not rebuild IONEX data; it expects a processed parquet file.


In [ ]:
!pip uninstall -y transformers scikit-learn scipy
!pip install -U accelerate bitsandbytes torchvision pillow sentencepiece protobuf safetensors huggingface_hub pandas pyarrow pydantic python-dateutil
!pip install "transformers @ git+https://github.com/huggingface/transformers.git@main"


IMPORTANT:

After this cell finishes, restart the Colab runtime:

Runtime -> Restart runtime

Do not import transformers before restarting.


## 2. Import Check

Do not import scipy/sklearn in this notebook.


In [ ]:
import torch
import transformers

print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
print("transformers Qwen imports OK")


## 3. CONFIG

For normal runs, change `RUN_ALL_TASKS` and `SELECTED_PRESET`, or edit the five task dictionaries below.

The interval convention is `[START, END)`: `START` is included and `END` is excluded.


In [ ]:
MODEL_NAME = "Qwen/Qwen3.5-4B"
LOAD_IN_4BIT = False
LOAD_IN_8BIT = False
TORCH_DTYPE = "float16"
MAX_INPUT_TOKENS = 4096
MAX_NEW_TOKENS = 512
TEMPERATURE = 0.0
DO_SAMPLE = False

DATASET_REF = "default"
DATASET_FILENAME = "tec_regions_2024_01_01_to_2024_04_01_hourly.parquet"
PROCESSED_CSV_PATH = None

START = "2024-03-01"
END = "2024-04-01"  # exclusive end

STATE_FEEDBACK_MODE = "state_aware"
MAX_STEPS = 14
MAX_TOOL_CALLS = 12
MAX_PARSE_RETRIES = 2
MAX_TOOL_RETRIES = 2

RUN_ALL_TASKS = True
SELECTED_PRESET = "stable_midlat_europe"

TASK_CONFIGS = [
    {
        "preset_id": "hightec_midlat_europe",
        "task_type": "high_tec",
        "region": "midlat_europe",
        "regions": ["midlat_europe"],
        "start": START,
        "end": END,
        "q": 0.90,
        "dataset_ref": DATASET_REF,
    },
    {
        "preset_id": "stable_midlat_europe",
        "task_type": "stable_intervals",
        "region": "midlat_europe",
        "regions": ["midlat_europe"],
        "start": START,
        "end": END,
        "window_minutes": 180,
        "q_delta": 0.60,
        "q_std": 0.60,
        "dataset_ref": DATASET_REF,
    },
    {
        "preset_id": "compare_midlat_europe_highlat_north",
        "task_type": "compare_regions",
        "regions": ["midlat_europe", "highlat_north"],
        "start": START,
        "end": END,
        "dataset_ref": DATASET_REF,
        "metrics": ["mean", "median", "min", "max", "std", "p90", "p95"],
    },
    {
        "preset_id": "compare_three_regions",
        "task_type": "compare_regions",
        "regions": ["equatorial_atlantic", "midlat_europe", "highlat_north"],
        "start": START,
        "end": END,
        "dataset_ref": DATASET_REF,
        "metrics": ["mean", "median", "min", "max", "std", "p90", "p95"],
    },
    {
        "preset_id": "report_midlat_europe",
        "task_type": "report",
        "region": "midlat_europe",
        "regions": ["midlat_europe"],
        "start": START,
        "end": END,
        "dataset_ref": DATASET_REF,
        "q": 0.90,
        "window_minutes": 180,
        "q_delta": 0.60,
        "q_std": 0.60,
        "include": ["basic_stats", "high_tec", "stable_intervals"],
        "metrics": ["mean", "median", "min", "max", "std", "p90", "p95"],
    },
]

MODEL_CONFIG = {
    "model_name": MODEL_NAME,
    "load_in_4bit": LOAD_IN_4BIT,
    "load_in_8bit": LOAD_IN_8BIT,
    "torch_dtype": TORCH_DTYPE,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "max_new_tokens": MAX_NEW_TOKENS,
    "temperature": TEMPERATURE,
    "do_sample": DO_SAMPLE,
}

AGENT_CONFIG = {
    "state_feedback_mode": STATE_FEEDBACK_MODE,
    "max_steps": MAX_STEPS,
    "max_tool_calls": MAX_TOOL_CALLS,
    "max_parse_retries": MAX_PARSE_RETRIES,
    "max_tool_retries": MAX_TOOL_RETRIES,
    "temperature": TEMPERATURE,
}

if RUN_ALL_TASKS:
    SELECTED_TASK_CONFIGS = list(TASK_CONFIGS)
else:
    SELECTED_TASK_CONFIGS = [
        item for item in TASK_CONFIGS if item["preset_id"] == SELECTED_PRESET
    ]
    assert SELECTED_TASK_CONFIGS, f"Unknown SELECTED_PRESET: {SELECTED_PRESET}"

print("RUN_ALL_TASKS:", RUN_ALL_TASKS)
print("selected presets:", [item["preset_id"] for item in SELECTED_TASK_CONFIGS])


## 4. Clone Or Update Repository


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path("/content/tec_agent_project")
REPO_URL = "https://github.com/ilyakosilov/tec_agent_project.git"

if not PROJECT_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(PROJECT_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(PROJECT_DIR), "pull", "origin", "main"], check=True)

os.chdir(PROJECT_DIR)

SRC_DIR = PROJECT_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("Working directory:", Path.cwd())
print("Latest commit:")
subprocess.run(["git", "log", "--oneline", "-3"], check=False)


## 5. Experiment Helpers

These functions are driven by each task config. Expected tool sequences are used only by evaluation; they are never sent to the agent as a next-tool instruction.


In [ ]:
from datetime import date

EPS = 1e-9


def _join_regions(region_ids):
    region_ids = list(region_ids)
    if len(region_ids) == 1:
        return region_ids[0]
    if len(region_ids) == 2:
        return f"{region_ids[0]} and {region_ids[1]}"
    return ", ".join(region_ids[:-1]) + f", and {region_ids[-1]}"


def build_query(config):
    task_type = config["task_type"]
    start = config["start"]
    end = config["end"]
    if task_type == "high_tec":
        return (
            f"Find high TEC intervals for {config['region']} "
            f"from {start} to {end} using q={config.get('q', 0.9):.2f} threshold."
        )
    if task_type == "stable_intervals":
        return (
            f"Find stable TEC intervals for {config['region']} "
            f"from {start} to {end} using the configured stability parameters."
        )
    if task_type == "compare_regions":
        return (
            f"Compare TEC statistics for {_join_regions(config['regions'])} "
            f"from {start} to {end}."
        )
    if task_type == "report":
        return (
            f"Build a concise TEC analysis report for {config['region']} "
            f"from {start} to {end}. Include high TEC intervals, stable intervals, "
            "key statistics, and a short interpretation based only on computed artifacts."
        )
    raise ValueError(f"Unsupported task_type: {task_type}")


def build_expected_tool_sequence(config):
    task_type = config["task_type"]
    if task_type == "high_tec":
        return [
            "tec_get_timeseries",
            "tec_compute_high_threshold",
            "tec_detect_high_intervals",
        ]
    if task_type == "stable_intervals":
        return [
            "tec_get_timeseries",
            "tec_compute_stability_thresholds",
            "tec_detect_stable_intervals",
        ]
    if task_type == "compare_regions":
        n_regions = len(config["regions"])
        return (
            ["tec_get_timeseries"] * n_regions
            + ["tec_compute_series_stats"] * n_regions
            + ["tec_compare_stats"]
        )
    if task_type == "report":
        return [
            "tec_get_timeseries",
            "tec_compute_series_stats",
            "tec_compute_high_threshold",
            "tec_detect_high_intervals",
            "tec_compute_stability_thresholds",
            "tec_detect_stable_intervals",
        ]
    raise ValueError(f"Unsupported task_type for expected sequence: {task_type}")


def build_eval_task(config, query, expected_sequence):
    task_type = config["task_type"]
    params = {}
    for key in ["metrics", "include", "window_minutes", "q_delta", "q_std"]:
        if key in config:
            params[key] = config[key]
    common = dict(
        task_id=f"qwen_smoke_{config['preset_id']}",
        query=query,
        task_type=task_type,
        dataset_ref=config["dataset_ref"],
        start=config["start"],
        end=config["end"],
        q=config.get("q", 0.9),
        expected_tool_sequence=tuple(expected_sequence),
        params=params or None,
    )
    if task_type in {"high_tec", "stable_intervals", "report"}:
        return EvalTask(
            **common,
            region_id=config.get("region"),
            region_ids=tuple(config.get("regions") or [config.get("region")]),
        )
    if task_type == "compare_regions":
        return EvalTask(
            **common,
            region_id=None,
            region_ids=tuple(config["regions"]),
        )
    raise ValueError(f"Unsupported task_type for EvalTask: {task_type}")


def agent_result_for_metrics(result):
    payload = result.tool_results
    if isinstance(payload, dict):
        payload = dict(payload)
        payload["final_answer"] = result.answer
    return payload


def print_key_metrics(task_type, metrics):
    common_keys = [
        "tool_sequence_match", "tool_call_count", "tool_error_count",
        "legacy_report_tool_used", "legacy_report_tool_absent",
        "start_date_match", "end_date_match", "date_parse_match",
        "expected_n_points", "agent_timeseries_n_points", "gold_timeseries_n_points",
        "timeseries_n_points_match", "region_parse_match", "region_set_match",
        "expected_region_id", "expected_region_ids", "actual_region_id",
        "actual_region_ids", "actual_region_source", "final_answer_present",
    ]
    task_keys = {
        "high_tec": [
            "threshold_abs_error", "threshold_exact_match", "interval_count_error",
            "interval_count_match", "top_interval_start_match", "top_interval_end_match",
            "global_peak_abs_error", "global_peak_match",
        ],
        "stable_intervals": [
            "stable_interval_count_error", "stable_interval_count_match",
            "top_stable_interval_start_match", "top_stable_interval_end_match",
            "top_stable_duration_abs_error", "top_stable_mean_abs_error",
            "top_stable_std_abs_error",
        ],
        "compare_regions": [
            "agent_region_count", "gold_region_count", "shared_region_count",
            "compare_stats_present", "stats_tool_call_count", "compare_stats_region_count",
            "pairwise_delta_count", "expected_pairwise_delta_count",
            "pairwise_delta_count_match", "pairwise_delta_region_pair_match",
            "mean_abs_error_avg", "max_abs_error_avg", "p90_abs_error_avg",
            "mean_abs_error_max", "max_abs_error_max", "p90_abs_error_max",
            "mean_delta_abs_error_avg", "max_delta_abs_error_avg", "p90_delta_abs_error_avg",
            "mean_delta_abs_error_max", "max_delta_abs_error_max", "p90_delta_abs_error_max",
        ],
        "report": [
            "report_present", "report_required_sections_present", "report_basic_stats_present",
            "report_high_tec_present", "report_stable_intervals_present",
            "required_artifacts_present", "missing_required_artifacts",
            "report_grounded_in_artifacts", "report_mean_abs_error_avg",
            "report_max_abs_error_avg", "report_p90_abs_error_avg",
            "report_high_tec_threshold_abs_error_avg",
            "report_high_tec_interval_count_error_avg",
            "report_stable_interval_count_error_avg",
        ],
    }.get(task_type, [])
    for key in common_keys + task_keys:
        print(f"{key}:", metrics.get(key))


def _metric_is_zero(metrics, key):
    value = metrics.get(key)
    return value is not None and abs(float(value)) <= EPS


def build_verdict_checks(task_type, result, gold_metric_result, metrics, expected_sequence):
    actual_sequence = list(result.tool_sequence)
    checks = {
        "no_runtime_error": result.error_message is None,
        "final_answer_present": bool(result.answer),
        "agent_success": bool(result.success),
        "tool_sequence_match": actual_sequence == expected_sequence and metrics.get("tool_sequence_match") is True,
        "no_tool_errors": metrics.get("tool_error_count") in {0, None},
        "no_parse_errors": result.parse_error_count == 0,
        "no_stall": not getattr(result, "stalled_loop_detected", False),
        "no_artifact_failure": not getattr(result, "artifact_usage_failure", False),
        "gold_metric_success": bool(gold_metric_result.success),
        "no_legacy_tool_used": metrics.get("legacy_report_tool_used") is not True,
    }
    if task_type == "high_tec":
        checks.update({
            "threshold_exact_match": metrics.get("threshold_exact_match") is True,
            "interval_count_match": metrics.get("interval_count_match") is True,
            "global_peak_match": metrics.get("global_peak_match") is True,
        })
    elif task_type == "stable_intervals":
        checks.update({
            "stable_interval_count_match": metrics.get("stable_interval_count_match") is True,
        })
    elif task_type == "compare_regions":
        checks.update({
            "region_set_match": metrics.get("region_set_match") is True,
            "pairwise_delta_count_match": metrics.get("pairwise_delta_count_match") is True,
            "compare_stats_present": metrics.get("compare_stats_present") is True,
            "mean_abs_error_avg_zero": _metric_is_zero(metrics, "mean_abs_error_avg"),
            "mean_abs_error_max_zero": _metric_is_zero(metrics, "mean_abs_error_max"),
            "p90_abs_error_avg_zero": _metric_is_zero(metrics, "p90_abs_error_avg"),
        })
    elif task_type == "report":
        checks.update({
            "report_present": metrics.get("report_present") is True,
            "required_artifacts_present": metrics.get("required_artifacts_present") is True,
            "report_grounded_in_artifacts": metrics.get("report_grounded_in_artifacts") is True,
            "high_intervals_artifact_present": metrics.get("report_high_tec_present") is True,
            "stable_intervals_artifact_present": metrics.get("report_stable_intervals_present") is True,
            "profile_or_stats_artifact_present": metrics.get("report_basic_stats_present") is True,
        })
    return checks


def compact_record(result):
    return {
        "answer": result.answer,
        "success": result.success,
        "error_message": result.error_message,
        "tool_sequence": result.tool_sequence,
        "parse_error_count": result.parse_error_count,
        "invalid_json_count": result.invalid_json_count,
        "unknown_format_count": result.unknown_format_count,
        "repair_attempt_count": result.repair_attempt_count,
        "repeated_tool_call_count": getattr(result, "repeated_tool_call_count", None),
        "stalled_loop_detected": getattr(result, "stalled_loop_detected", None),
        "artifact_usage_failure": getattr(result, "artifact_usage_failure", None),
        "malformed_wrapper_count": getattr(result, "malformed_wrapper_count", None),
        "cleaned_output_changed_count": getattr(result, "cleaned_output_changed_count", None),
        "available_artifacts": getattr(result, "available_artifacts", {}),
        "missing_goal_artifacts": getattr(result, "missing_goal_artifacts", []),
        "completed_tool_calls": getattr(result, "completed_tool_calls", []),
        "state_snapshots": getattr(result, "state_snapshots", []),
        "trace": result.trace,
        "tool_results": result.tool_results,
        "raw_model_outputs": getattr(result, "raw_model_outputs", []),
        "cleaned_model_outputs": getattr(result, "cleaned_model_outputs", []),
    }


## 6. Dataset Setup

Required processed dataset is selected by `DATASET_FILENAME` in CONFIG.

This notebook does not rebuild IONEX data. Use `notebooks/01_build_tec_dataset.ipynb` to build the processed parquet, then copy it here.


In [ ]:
from pathlib import Path
import shutil

DATASET_PATH = PROJECT_DIR / "data" / "processed" / DATASET_FILENAME
if PROCESSED_CSV_PATH:
    DATASET_PATH = Path(PROCESSED_CSV_PATH)
DATASET_PATH.parent.mkdir(parents=True, exist_ok=True)

print("Expected dataset:", DATASET_PATH)
print("Exists:", DATASET_PATH.exists())
print("Dataset ref:", DATASET_REF)
print("Analysis period:", f"[{START}, {END})")


If the dataset is in Google Drive, uncomment and edit the next cell.


In [ ]:
# from google.colab import drive
# drive.mount("/content/drive")
#
# DRIVE_PARQUET_PATH = Path("/content/drive/MyDrive") / DATASET_FILENAME
# shutil.copy2(DRIVE_PARQUET_PATH, DATASET_PATH)
# print("Copied:", DATASET_PATH.exists(), DATASET_PATH)


## 7. Project Imports And Dataset Registration


In [ ]:
from tec_agents.agents.llm_single_agent import LLMSingleAgent
from tec_agents.data.datasets import get_dataset_summary, register_dataset
from tec_agents.eval.gold_runner import GoldRunner
from tec_agents.eval.metrics import compare_agent_to_gold
from tec_agents.eval.task_set import EvalTask, task_to_dict
from tec_agents.llm.local_qwen import LocalQwenChatModel
from tec_agents.mcp.client import LocalMCPClient
from tec_agents.mcp.server import build_local_mcp_server

assert DATASET_PATH.exists(), (
    f"Missing dataset: {DATASET_PATH}. "
    "Build it with notebooks/01_build_tec_dataset.ipynb or copy it from Drive."
)

register_dataset(
    dataset_ref=DATASET_REF,
    path=DATASET_PATH,
    file_format="parquet" if str(DATASET_PATH).endswith(".parquet") else "csv",
)

summary = get_dataset_summary(DATASET_REF)
summary


## 8. Initialize MCP-like Tools


In [ ]:
tool_check_server = build_local_mcp_server(run_id="qwen_single_agent_batch_tool_check")
tool_check_client = LocalMCPClient(tool_check_server)

tool_names = tool_check_client.list_tool_names()
print("Available tools:")
for name in tool_names:
    print("-", name)

required = {
    "tec_get_timeseries",
    "tec_compute_high_threshold",
    "tec_detect_high_intervals",
    "tec_compute_series_stats",
    "tec_compare_stats",
    "tec_compute_stability_thresholds",
    "tec_detect_stable_intervals",
}

print("Missing required tools:", sorted(required - set(tool_names)))
print("Legacy report tool present:", "tec_build_report" in tool_names)

assert "tec_build_report" not in tool_names
for cfg in TASK_CONFIGS:
    seq = build_expected_tool_sequence(cfg)
    assert "tec_build_report" not in seq
    assert "tec_compare_regions" not in seq


## 9. Optional Hugging Face Login


In [ ]:
# Optional: use a Hugging Face token for higher rate limits or gated models.
# In Colab, add a secret named HF_TOKEN or HF_read_token.
try:
    from google.colab import userdata
    from huggingface_hub import login

    hf_token = userdata.get("HF_TOKEN") or userdata.get("HF_read_token")
    if hf_token:
        login(token=hf_token)
        print("Logged in to Hugging Face.")
    else:
        print("No HF token found. Continuing unauthenticated.")
except Exception as exc:
    print("HF login skipped:", exc)


## 10. Load Qwen

The model is loaded once, then reused for all selected tasks. Default uses float16 without bitsandbytes 4-bit because some Colab runtimes fail with `libnvJitLink.so.13`.


In [ ]:
model = LocalQwenChatModel(
    model_name=MODEL_CONFIG["model_name"],
    device_map="auto",
    torch_dtype=MODEL_CONFIG["torch_dtype"],
    load_in_4bit=MODEL_CONFIG["load_in_4bit"],
    load_in_8bit=MODEL_CONFIG["load_in_8bit"],
    trust_remote_code=True,
    max_input_tokens=MODEL_CONFIG["max_input_tokens"],
)


## 11. Batch Run

Each task gets a fresh MCP-like server/client so traces and artifacts are isolated. The expected sequence is used only after the run for evaluation.


In [ ]:
import json

all_results = []
output_dir = PROJECT_DIR / "outputs" / "metrics"
output_dir.mkdir(parents=True, exist_ok=True)

gold_runner = GoldRunner()

for index, task_config in enumerate(SELECTED_TASK_CONFIGS, start=1):
    preset_id = task_config["preset_id"]
    task_type = task_config["task_type"]
    print("=" * 100)
    print(f"TASK {index}/{len(SELECTED_TASK_CONFIGS)}: {preset_id} ({task_type})")

    query = build_query(task_config)
    expected_sequence = build_expected_tool_sequence(task_config)
    eval_task = build_eval_task(task_config, query, expected_sequence)

    print("query:", query)
    print("expected_sequence:", expected_sequence)
    print("analysis_period:", f"[{task_config['start']}, {task_config['end']})")

    server = build_local_mcp_server(run_id=f"qwen_single_agent_{preset_id}_colab")
    client = LocalMCPClient(server)
    agent = LLMSingleAgent(
        model=model,
        client=client,
        max_steps=AGENT_CONFIG["max_steps"],
        max_tool_calls=AGENT_CONFIG["max_tool_calls"],
        max_parse_retries=AGENT_CONFIG["max_parse_retries"],
        max_tool_retries=AGENT_CONFIG["max_tool_retries"],
        temperature=AGENT_CONFIG["temperature"],
        tool_max_new_tokens=min(256, int(MODEL_CONFIG["max_new_tokens"])),
        final_max_new_tokens=int(MODEL_CONFIG["max_new_tokens"]),
        state_feedback_mode=AGENT_CONFIG["state_feedback_mode"],
    )

    result = agent.run(query)
    actual_sequence = list(result.tool_sequence)
    sequence_match = actual_sequence == expected_sequence

    gold_result = gold_runner.run(eval_task)
    print("gold_status:", gold_result.status)
    print("gold_error:", gold_result.error)

    if gold_result.status == "success" and gold_result.result is not None:
        metric_result = compare_agent_to_gold(
            task_id=eval_task.task_id,
            task_type=eval_task.task_type,
            agent_result=agent_result_for_metrics(result),
            gold_result=gold_result.result,
            agent_trace=result.trace,
            task=task_to_dict(eval_task),
            parsed_task=result.parsed_task,
            orchestration_steps=result.orchestration_steps,
        )
        metrics = metric_result.metrics
    else:
        metric_result = None
        metrics = {}

    verdict_checks = build_verdict_checks(
        task_type,
        result,
        metric_result if metric_result is not None else type("MetricStub", (), {"success": False})(),
        metrics,
        expected_sequence,
    )
    overall_ok = all(verdict_checks.values())

    print("agent_success:", result.success)
    print("error_message:", result.error_message)
    print("actual_sequence:", actual_sequence)
    print("sequence_match:", sequence_match)
    print("tool_call_count:", result.trace.get("n_calls"))
    print("parse_error_count:", result.parse_error_count)
    print("malformed_wrapper_count:", getattr(result, "malformed_wrapper_count", None))
    print_key_metrics(task_type, metrics)
    print("verdict_checks:")
    for key, value in verdict_checks.items():
        print(f"  {key}: {value}")
    print("OVERALL_OK:", overall_ok)
    print("final_answer_preview:")
    print((result.answer or "")[:1200])

    record = {
        "selected_preset": preset_id,
        "task_config": task_config,
        "query": query,
        "expected_tool_sequence": expected_sequence,
        "actual_tool_sequence": actual_sequence,
        "agent_result": compact_record(result),
        "gold_status": gold_result.status,
        "gold_error": gold_result.error,
        "gold_result": gold_result.result,
        "metrics": metrics,
        "metric_success": None if metric_result is None else metric_result.success,
        "metric_errors": [] if metric_result is None else metric_result.errors,
        "verdict_checks": verdict_checks,
        "success": overall_ok,
    }

    per_task_path = output_dir / f"qwen_single_agent_{preset_id}_colab.json"
    with per_task_path.open("w", encoding="utf-8") as f:
        json.dump(record, f, ensure_ascii=False, indent=2, default=str)
    print("saved_per_task_json:", per_task_path)

    all_results.append(record)

batch_payload = {
    "model_config": MODEL_CONFIG,
    "agent_config": AGENT_CONFIG,
    "dataset_ref": DATASET_REF,
    "dataset_path": str(DATASET_PATH),
    "run_all_tasks": RUN_ALL_TASKS,
    "selected_preset": SELECTED_PRESET,
    "results": all_results,
}

batch_path = output_dir / "qwen_single_agent_batch_colab.json"
with batch_path.open("w", encoding="utf-8") as f:
    json.dump(batch_payload, f, ensure_ascii=False, indent=2, default=str)
print("saved_batch_json:", batch_path)


## 12. Summary Table


In [ ]:
summary_rows = []
for record in all_results:
    metrics = record.get("metrics") or {}
    summary_rows.append(
        {
            "preset_id": record["selected_preset"],
            "task_type": record["task_config"]["task_type"],
            "success": record["success"],
            "tool_sequence_match": metrics.get("tool_sequence_match"),
            "tool_call_count": metrics.get("tool_call_count"),
            "tool_error_count": metrics.get("tool_error_count"),
            "key_metric": (
                metrics.get("interval_count_match")
                if record["task_config"]["task_type"] == "high_tec"
                else metrics.get("stable_interval_count_match")
                if record["task_config"]["task_type"] == "stable_intervals"
                else metrics.get("region_set_match")
                if record["task_config"]["task_type"] == "compare_regions"
                else metrics.get("report_required_sections_present")
            ),
            "answer_preview": (record["agent_result"].get("answer") or "")[:160],
        }
    )

try:
    import pandas as pd
    summary_df = pd.DataFrame(summary_rows)
    display(summary_df)
except Exception:
    for row in summary_rows:
        print(row)


## 13. Inspect Raw And Cleaned Model Outputs


In [ ]:
for record in all_results:
    print("=" * 100)
    print(record["selected_preset"])
    agent_result = record["agent_result"]
    for i, raw in enumerate(agent_result.get("raw_model_outputs", []), start=1):
        print("-" * 80)
        print("RAW MODEL OUTPUT", i)
        print(raw)
    for i, cleaned in enumerate(agent_result.get("cleaned_model_outputs", []), start=1):
        print("-" * 80)
        print("CLEANED MODEL OUTPUT", i)
        print(cleaned)


## 14. Optional Download Result


In [ ]:
# from google.colab import files
# files.download(str(batch_path))
